# BNP Paribas Personal Finance - Fraud Detection

## Summary

Fraud is a major problem for merchants. Criminals use a wide variety of methods to attack organizations across systems, channels, processes and products. The development of fraud detection methods is thus of crucial importance. Fraud detection is a challenging problem because fraudsters make their best efforts to make their behaviour look legitimate. Another difficulty is that the number of legitimate records is far greater than the number of fraudulent cases.


In [12]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from skopt import BayesSearchCV

from sklearn.linear_model import Perceptron

In [13]:
df = pd.read_csv("../data/fraud/FraudeCanastas.csv", index_col="ID")

df.head()

,APPLE PRODUCTDESCRIPTION | SAMSUNG | MODEL90,AUDIO ACCESSORIES | AB AUDIO | AB AUDIO GO AIR TRUE WIRELESS BLUETOOTH IN-EAR H,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE 2ND GENERATI,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH WIRELESS CHARGING CASE,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH WIRELESS CHARGING CASE 2ND,AUDIO ACCESSORIES | APPLE | 2021 APPLE AIRPODS WITH MAGSAFE CHARGING CASE 3RD,AUDIO ACCESSORIES | APPLE | AIRPODS PRO,AUDIO ACCESSORIES | APPLE | APPLE AIRPODS MAX,AUDIO ACCESSORIES | APPLE | APPLE AIRPODS MAX NOISE CANCELLING WIRELESS BLUETO,...,WOMEN S NIGHTWEAR | ANYDAY RETAILER | ANYDAY RETAILER LEOPARD PRINT JERSEY PY,WOMEN S NIGHTWEAR | RETAILER | RETAILER CLEO VELOUR JOGGER LOUNGE PANT,WOMEN S NIGHTWEAR | SOSANDAR | SOSANDAR ZEBRA PRINT PYJAMA BOTTOMS BLACK 10,Nb_of_items,total_of_items,costo_total,costo_medio_item,costo_item_max,costo_item_min,fraud_flag
ID,,,,,,,,,,,,,,,,,,,,,
130,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2,2,1299,649.500000,1299,0.0,1.0
195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3,3,4119,1373.000000,2470,0.0,1.0
217,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2,2,2806,1403.000000,2799,7.0,1.0
552,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2,2,1206,603.000000,1199,7.0,1.0
854,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,19,27,1807,66.925926,195,4.0,1.0


In [14]:
df.info()

<class 'pandas.DataFrame'>
Index: 9319 entries, 130 to 49022
Columns: 2456 entries, APPLE PRODUCTDESCRIPTION | SAMSUNG | MODEL90 to fraud_flag
dtypes: float64(2452), int64(4)
memory usage: 174.7 MB


In [15]:
def describe(df):

    unique = []
    for col in df.columns:
        unique.append(df[col].unique())
    unique_series = pd.Series(unique, index=df.columns)
    return pd.concat(
        [
            df.dtypes,
            df.isna().sum(),
            round(df.isna().sum() / len(df) * 100, 1),
            df.nunique(),
            unique_series,
        ],
        axis=1,
        keys=["dtype", "missing", "%null", "nunique", "unique"],
    )


describe(df)

,dtype,missing,%null,nunique,unique
APPLE PRODUCTDESCRIPTION | SAMSUNG | MODEL90,float64,0,0.0,2,"[0.0, 1000.0]"
AUDIO ACCESSORIES | AB AUDIO | AB AUDIO GO AIR TRUE WIRELESS BLUETOOTH IN-EAR H,float64,0,0.0,2,"[0.0, 20.0]"
AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE,float64,0,0.0,8,"[0.0, 125.0, 119.0, 120.0, 500.0, 129.0, 109.0..."
AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE 2ND GENERATI,float64,0,0.0,8,"[0.0, 109.0, 100.0, 105.0, 104.0, 99.0, 119.0,..."
AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH WIRELESS CHARGING CASE,float64,0,0.0,4,"[0.0, 149.0, 158.0, 168.0]"
...,...,...,...,...,...
costo_total,int64,0,0.0,1639,"[1299, 4119, 2806, 1206, 1807, 1263, 942, 1199..."
costo_medio_item,float64,0,0.0,2034,"[649.5, 1373.0, 1403.0, 603.0, 66.925925925925..."
costo_item_max,int64,0,0.0,540,"[1299, 2470, 2799, 1199, 195, 280, 938, 929, 1..."
costo_item_min,float64,0,0.0,528,"[0.0, 7.0, 4.0, 1249.0, 25.0, 2470.0, 999.0, 1..."


In [16]:
basket = df.iloc[:, :-7].astype(int)
total = df.iloc[:, -7:-1]
fraud = df.iloc[:, -1]

In [ ]:
encoder = OneHotEncoder(drop="first", sparse_output=False)
one_hot_basket = encoder.fit_transform(basket)

one_hot_basket_df = pd.DataFrame(
    one_hot_basket,
    index=basket.index,
    columns=encoder.get_feature_names_out(),
).astype(int)

describe(one_hot_basket_df)

,dtype,missing,%null,nunique,unique
APPLE PRODUCTDESCRIPTION | SAMSUNG | MODEL90_1000,int64,0,0.0,2,"[0, 1]"
AUDIO ACCESSORIES | AB AUDIO | AB AUDIO GO AIR TRUE WIRELESS BLUETOOTH IN-EAR H_20,int64,0,0.0,2,"[0, 1]"
AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_109,int64,0,0.0,2,"[0, 1]"
AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_119,int64,0,0.0,2,"[0, 1]"
AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_120,int64,0,0.0,2,"[0, 1]"
...,...,...,...,...,...
WOMEN S FOOTWEAR | ROCKET DOG | ROCKET DOG CHEERY CANVAS TRAINERS WHITE 3_38,int64,0,0.0,2,"[0, 1]"
WOMEN S FOOTWEAR | UGG | UGG CLASSIC MINI II SHORT SHEEPSKIN BOOTS CHESTNUT_145,int64,0,0.0,2,"[0, 1]"
WOMEN S NIGHTWEAR | ANYDAY RETAILER | ANYDAY RETAILER LEOPARD PRINT JERSEY PY_28,int64,0,0.0,2,"[0, 1]"
WOMEN S NIGHTWEAR | RETAILER | RETAILER CLEO VELOUR JOGGER LOUNGE PANT_18,int64,0,0.0,2,"[0, 1]"


KeyboardInterrupt: 

In [ ]:
vectorizer = TfidfTransformer()

vectorized_basket = vectorizer.fit_transform(one_hot_basket_df)

vectorized_basket_df = pd.DataFrame(
    vectorized_basket.todense(),  # type: ignore
    index=basket.index,
    columns=vectorizer.get_feature_names_out(),
)

describe(vectorized_basket_df)

,dtype,missing,%null,nunique,unique
APPLE PRODUCTDESCRIPTION | SAMSUNG | MODEL90_1000,float64,0,0.0,2,"[0.0, 0.7071067811865475]"
AUDIO ACCESSORIES | AB AUDIO | AB AUDIO GO AIR TRUE WIRELESS BLUETOOTH IN-EAR H_20,float64,0,0.0,2,"[0.0, 0.7244356844669503]"
AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_109,float64,0,0.0,3,"[0.0, 0.7886436942612468, 0.7684392618067257]"
AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_119,float64,0,0.0,2,"[0.0, 0.9120216669061357]"
AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_120,float64,0,0.0,3,"[0.0, 0.6182557237683995, 0.583472366450739]"
...,...,...,...,...,...
WOMEN S FOOTWEAR | ROCKET DOG | ROCKET DOG CHEERY CANVAS TRAINERS WHITE 3_38,float64,0,0.0,2,"[0.0, 0.2791597618386579]"
WOMEN S FOOTWEAR | UGG | UGG CLASSIC MINI II SHORT SHEEPSKIN BOOTS CHESTNUT_145,float64,0,0.0,2,"[0.0, 0.3694019711388358]"
WOMEN S NIGHTWEAR | ANYDAY RETAILER | ANYDAY RETAILER LEOPARD PRINT JERSEY PY_28,float64,0,0.0,2,"[0.0, 0.3278908480268799]"
WOMEN S NIGHTWEAR | RETAILER | RETAILER CLEO VELOUR JOGGER LOUNGE PANT_18,float64,0,0.0,2,"[0.0, 0.29301947952106516]"


In [ ]:
scaler = MinMaxScaler()
total_scaled = scaler.fit_transform(total)
total_scaled_df = pd.DataFrame(
    total_scaled,
    index=total.index,
    columns=scaler.get_feature_names_out(),
)
describe(total_scaled_df)

,dtype,missing,%null,nunique,unique
Nb_of_items,float64,0,0.0,28,"[0.02702702702702703, 0.05405405405405406, 0.4..."
total_of_items,float64,0,0.0,34,"[0.023255813953488372, 0.046511627906976744, 0..."
costo_total,float64,0,0.0,1639,"[0.044385993753082366, 0.16028275521946408, 0...."
costo_medio_item,float64,0,0.0,2034,"[0.03045913215081366, 0.0667113773193651, 0.06..."
costo_item_max,float64,0,0.0,540,"[0.0588353716845337, 0.11777140268760382, 0.13..."
costo_item_min,float64,0,0.0,528,"[0.0, 0.00035001750087504374, 0.00020001000050..."


In [ ]:
processed_df = vectorized_basket_df.join(total_scaled_df)
processed_df.head()

,APPLE PRODUCTDESCRIPTION | SAMSUNG | MODEL90_1000,AUDIO ACCESSORIES | AB AUDIO | AB AUDIO GO AIR TRUE WIRELESS BLUETOOTH IN-EAR H_20,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_109,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_119,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_120,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_125,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_129,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_130,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE_500,AUDIO ACCESSORIES | APPLE | 2019 APPLE AIRPODS WITH CHARGING CASE 2ND GENERATI_99,...,WOMEN S FOOTWEAR | UGG | UGG CLASSIC MINI II SHORT SHEEPSKIN BOOTS CHESTNUT_145,WOMEN S NIGHTWEAR | ANYDAY RETAILER | ANYDAY RETAILER LEOPARD PRINT JERSEY PY_28,WOMEN S NIGHTWEAR | RETAILER | RETAILER CLEO VELOUR JOGGER LOUNGE PANT_18,WOMEN S NIGHTWEAR | SOSANDAR | SOSANDAR ZEBRA PRINT PYJAMA BOTTOMS BLACK 10_26,Nb_of_items,total_of_items,costo_total,costo_medio_item,costo_item_max,costo_item_min
ID,,,,,,,,,,,,,,,,,,,,,
130,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.027027,0.023256,0.044386,0.030459,0.058835,0.00000
195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.054054,0.046512,0.160283,0.066711,0.117771,0.00000
217,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.027027,0.023256,0.106321,0.068215,0.134330,0.00035
552,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.027027,0.023256,0.040564,0.028129,0.053802,0.00035
854,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.486486,0.604651,0.065264,0.001268,0.003271,0.00020


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    processed_df,
    fraud,
    test_size=0.15,
    shuffle=True,
    random_state=42,
)

len(X_train), len(X_test), len(y_train), len(y_test)

(7921, 1398, 7921, 1398)

In [ ]:
params = {
    "penalty": [None, "l2"],
    "alpha": np.logspace(-10, 1, 100),
    "tol": np.logspace(-10, 1, 100),
}

class_weights = compute_class_weight(
    y=fraud, class_weight="balanced", classes=np.unique(y_train)
)

class_weight_dict = dict(zip(np.unique(y_train), class_weights))

model1 = Perceptron(class_weight=class_weight_dict)

bayes_opt = BayesSearchCV(model1, params, n_iter=50, scoring="recall")
bayes_opt.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
print(f"Best Params: {bayes_opt.best_params_}")  # type: ignore
print(f"Score: {bayes_opt.score(X_test, y_test)}")

In [ ]:
y_pred = bayes_opt.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))